<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Simple Linear Regression with Sacramento Real Estate Data

_Authors: Matt Brems, Sam Stack_

---

In this lab you will hone your exploratory data analysis (EDA) skills and practice constructing simple linear regressions using a data set on Sacramento real estate sales.  The data set contains information on qualities of the property, location of the property, and time of sale.

### 1. Read in the Sacramento housing data set.

In [ ]:
sac_csv = './datasets/sacramento_real_estate_transactions.csv'

In [ ]:
import pandas as pd

shd = pd.read_csv(sac_csv)

shd.head()

### 2. Conduct exploratory data analysis on this data set. 

**Report any notable findings here and any steps you take to clean/process data.**

> **Note:** These EDA checks should be done on every data set you handle. If you find yourself checking repeatedly for missing/corrupted data, it might be beneficial to have a function that you can reuse every time you're given new data.

In [ ]:
# Check the dtypes.
shd.dtypes

In [ ]:
# Zip code works better as a string object (categorical) so we will have to convert it.
shd['zip'] = shd['zip'].astype(str)

_**Fun Fact:** Zip codes often have leading zeros — e.g., 02215 = Boston, MA — which will often get knocked off automatically by many software programs like Python or Excel. You can imagine that this could create some issues. _

In [ ]:
# Check out the summary statistics:
shd.describe()

In [ ]:
# It appears that we have some values that are out of place, as there are
# houses with 0 bedrooms, 0 baths, negative square footage, and negative prices.  
# There are also some bizarre longitudes/latitudes in our data set. A house in Antarctica perhaps? 

In [ ]:
# Check out the cities. Most cities have very few observations.
shd.city.value_counts()

In [ ]:
# What could explain the houses with 0 bedrooms?
shd[shd['beds'] == 0]

In [ ]:
print shd[shd['beds'] == 0].shape

In [ ]:
# Given the large number of houses that have 0 beds, 0 baths and 0 square feet, 
# we can make an assumption that these are plots of land that haven't had anything built on them yet.
# As a result we will *not* be dropping them.

In [ ]:
# What about the houses that are less than 0 dollars?
shd[shd['price'] < 1]

In [ ]:
# And the houses with negative square footage?
shd[shd['sq__ft'] < 0]

In [ ]:
# It looks like the house with a negative price is also the one with negative square footage.
# It is time to make a choice: Either assume that the data were entered improperly and are meant 
# to be positive, or drop the data altogether.

# Side note: Notice the state is actually labeled incorrectly as well.

In [ ]:
# Let's check if any other values are labeled incorrectly.
shd[shd['state'] != 'CA']

In [ ]:
# It looks like only one row is mislabeled, so we'll just drop that row.

shd.drop(703, inplace = True)

### 3. Our goal will be to predict price. List variables that you think qualify as predictors of price in an SLR model. 

**For each of the variables you believe to be a valid potential predictor in an SLR model, generate a plot showing the relationship between the independent and dependent variables.**

In [ ]:
# 'Beds','baths', and'sq__ft' seem like they would be valid potential predictors.
# 'Latitude' and 'longitude' are other options to consider, but those are more involved.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
% matplotlib inline

sns.lmplot(x='sq__ft', y='price', data=shd)
plt.show()
sns.lmplot(x='beds', y='price', data=shd)
plt.show()
sns.lmplot(x='baths', y='price', data=shd)
plt.show()

When you've finished cleaning or have made a good deal of progress cleaning, it's always a good idea to save your work.
```python
shd.to_csv('./datasets/sacramento_real_estate_transactions_Clean.csv')
```

### 4. Which variable would be the best predictor of Y in an SLR model? Why?

In [ ]:
# It looks like square footage is a better predictor of price than number of beds or baths.
# Beds and baths are discrete measures as opposed to square footage, which is a continuous measure.  
# Additionally, there are likely strong correlations between the square footage and numbers of bedrooms and bathrooms, in that houses 
# with greater square footage will likely have more bedrooms and bathrooms.

# If we dropped all of the plots of land that are in the data set (those with 0 sq ft, 
# beds, and baths), we would see a much stronger trend line in our lm plot.

### 5. Build a function that will take in two lists, `Y` and `X`, and return the intercept and slope coefficients that minimize SSE. 

`Y` is the target variable and `X` is the predictor variable.

- **Test your function on price and the variable you determined was the best predictor in Problem 4.**
- **Report the slope and intercept.**

In [ ]:
import numpy as np
import scipy.stats

# Get the optimal slope and y intercept.

def lin_reg(x,y):
    # Using other libraries for standard deviation and the Pearson correlation coefficient.
    # Note that in SLR, the correlation coefficient multiplied by the standard
    # deviation of y divided by standard deviation of x is the optimal slope.
    beta_1 = (scipy.stats.pearsonr(x,y)[0])*(np.std(y)/np.std(x))
    
    # The Pearson correlation coefficient returns a tuple, so it needs to be sliced/indexed.
    # The optimal beta is found by: mean(y) - b1 * mean(x).
    beta_0 = np.mean(y)-(beta_1*np.mean(x)) 
    
    #Print the optimal values.
    print 'The Optimal Y Intercept is ', beta_0
    print 'The Optimal slope is ', beta_1

In [ ]:
x = shd['sq__ft'].values
y = shd['price'].values
lin_reg(x,y)

### 6. Interpret the intercept. Interpret the slope.

In [ ]:
# The intercept should be the relative y value around which our data begin 
# in that from this point out, as the x values increase, so should the y value from this point.

# The intercept is an "offset." Without the intercept, our regression line would
# be forced to pass through the origin.

# The slope is the increase in our target variable (price) for every 1-unit increase
# in our predictor variable (sq__ft). So in our example, for every sq__ft increase there is
# an associated increase of ~54 dollars in price. 

### 7. Give an example of how this model could be used for prediction and how it could be used for inference. 

**Be sure to make it clear which example is associated with prediction and which is associated with inference.**

In [ ]:
# Prediction:
# You are a real estate agent with a separate database on house characteristics and locations.
# You want to estimate what price one of these houses will most likely sell for based
# on the model that we built using this data.

# Inference:
# You work for an architecture company that wants to understand what specific locations and housing characteristics 
# are associated with perceived value. You have some hypotheses about what
# makes a house valuable but you would like to test these hypotheses.

### 8: [Bonus] Using the model you came up with in Problem 5, calculate and plot the residuals.

In [ ]:
# Predictor: y = 162938.74 + 54.16x
# Creating a list of predicted values:
y_pred = []

for x in shd['sq__ft']:
    y = 162938.74 + (54.16*x)
    y_pred.append(y)


In [ ]:
# Appending the predicted values to the Sacramento housing DataFrame to do DF calcs:
shd['Pred'] = y_pred

# Residuals equals the difference between Y-True and Y-Pred:
shd['Residuals'] = abs(shd['price']-shd['Pred'])

In [ ]:
shd['Residuals'].mean()
# The mean of our residuals is approximately $96,000, which means that 
# on average our prediction is off by this amount.

In [ ]:
# Plot showing our linear forecast.
fig = plt.figure(figsize=(20,20))

# Change the font size of minor ticks label.
plot = fig.add_subplot(111)
plot.tick_params(axis='both', which='major', labelsize=20)

# Get the axis of that figure.
ax = plt.gca()

# Plot a scatterplot on the axis using our data.
ax.scatter(x= shd['sq__ft'], y=shd['price'], c='k')
ax.plot(shd['sq__ft'], shd['Pred'], color='r');

In [ ]:
# Plot with residuals.
fig = plt.figure(figsize=(20,20))

# Change the font size of minor ticks label.
plot = fig.add_subplot(111)
plot.tick_params(axis='both', which='major', labelsize=20)

# Get the axis of that figure.
ax = plt.gca()

# Plot a scatterplot on the axis using our data.
ax.scatter(x= shd['sq__ft'], y=shd['price'], c='k')
ax.plot(shd['sq__ft'], shd['Pred'], color='r');

# Iterate over predictions.
for _, row in shd.iterrows():
    plt.plot((row['sq__ft'], row['sq__ft']), (row['price'], row['Pred']), 'b-')

In [ ]:
# One more plot; let's look at how our predictions compare to the true values.
sns.lmplot(x='price', y='Pred', data=shd)

In [ ]:
# Given our last visual, we can see that a lot of points were plotted along the 
# y intercept (y= 162938.74). These were all the houses reported to have a
# square footage of 0. We could probably create a more insightful
# model if we removed those observations from our data.